# 05 Live 20-question Graph RAG evaluation (LangGraph agent on Neo4j, scored with the team benchmark)

**Date:** 2026-10-22
**Author:** Xincheng (George) Ding
**Needs network** when `IATI_LIVE_EVAL=1` (model APIs + Aura). Default is a dry run with a synthetic model and no database.

**Purpose.** Run the team's 20-question benchmark (`fa26/benchmark/`, by Deniz Özçakır) end to end with real models over the Neo4j graph loaded by notebook 04 v2. For every question the LangGraph agent decides whether to **answer, abstain or clarify**, writes one read-only Cypher query that returns `key / value / evidence`, executes it, retries on failure, and emits the benchmark's structured JSON. Submissions are scored by the benchmark's own `score.py` (never by an LLM), so correctness, evidence lineage and abstention behaviour are measured on the same scale as the SQL baseline.

**What is new vs notebook 03.** 03 compared *query results* against five hand-written gold queries. 05 compares *structured answers* against the benchmark answer key: 16 answerable questions (70% value correctness + 30% evidence F1) and 4 limitation questions (reason-code F1). Evidence is the list of `source_id`s (`<table>.csv:row:<n>`) behind each fact, which is why 04 v2 stamps every node and relationship with one.

**Outputs**
- `fa26/data/processed/05_submission_<model>_<run>.json` (one per model, benchmark submission format)
- `fa26/benchmark/results/live_<run>/comparison.md|csv, details.json` (scorer output)
- `fa26/data/processed/05_live_eval_<run>.csv` (long table: model × question × status / score / latency / cypher)

**Agent graph:** `plan → guard → execute → validate → format` with up to 2 retries that feed the failure back to `plan`. Abstain / clarify skips straight to `format`.

## V2 inputs and offline verification
Export a public v2 bundle, then set `IATI_BENCHMARK_BUNDLE` to its absolute directory before running. Development is the recommended first split. With no bundle, the original v1 questions are used. `IATI_LIVE_EVAL=0` runs fake models without a database; set `1` only for an authorized live run. V2 scoring runs after generation in a separate evaluator process. New questions supply no evaluator contracts or expected value types. Notebook 06 skips its hard-coded v1 manual routes for v2 and evaluates the existing LLM router plus fallback. No new templates have been tuned on held-out questions. Results include benchmark and split metadata; comparisons use only submissions from this run.


In [1]:
from pathlib import Path
import os, sys, re, json, time, subprocess
from decimal import Decimal
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fa26" / "benchmark" / "manifest.json").exists())
FA26 = ROOT / "fa26"
sys.path.insert(0, str(FA26 / "benchmark")); sys.path.insert(0, str(FA26 / "src"))
from common import verify_snapshot, load_json, save_json
from text2cypher_eval import guard_query, read_executor          # Deniz's reviewed guard and read-only executor
SNAPSHOT = verify_snapshot(); SNAPSHOT_ID = SNAPSHOT["snapshot_id"]

import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv(FA26 / ".env", override=False)
DRY_RUN = os.environ.get("IATI_LIVE_EVAL", "0") != "1"
RUN_ID = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S_%f")
PROCESSED = FA26 / "data" / "processed"; PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS = FA26 / "benchmark" / "results" / f"live_{RUN_ID}"
print("run", RUN_ID, "| dry run:", DRY_RUN, "| snapshot", SNAPSHOT_ID[:16])
from benchmark_bundle import load_bundle, submission, score_command, read_scores
sys.path.insert(0, str(FA26 / "src"))
from tools.contract import evidence_error, clean_evidence, normalise_facts, single_v2_reason, V2_REASON_GUIDE   # output contract (evidence ids, scalar keys, reasons)
BUNDLE = load_bundle(os.environ["IATI_BENCHMARK_BUNDLE"], SNAPSHOT_ID) if os.environ.get("IATI_BENCHMARK_BUNDLE") else None
if BUNDLE:
    RESULTS = FA26 / "benchmark" / "v2" / "results" / "live" / ("05_" + BUNDLE["template"]["split"] + "_" + RUN_ID)


run 2026-11-05_193242_484749 | dry run: False | snapshot 430caabd5a8ae1d0


## 1. Questions and contracts (public inputs only; the answer key is never loaded here)

In [2]:
QUESTIONS = load_json(FA26 / "benchmark" / "questions.json")
TEMPLATE = load_json(FA26 / "benchmark" / "submission_template.json")
assert TEMPLATE["snapshot_id"] == SNAPSHOT_ID
REASON_CODES = ["NEEDS_DATA_MISSING", "HISTORY_UNAVAILABLE", "FUNDING_DEFINITION_REQUIRED",
                "CURRENCY_BASIS_REQUIRED", "PERIOD_REQUIRED", "RAW_XML_UNAVAILABLE"]
if BUNDLE:
    QUESTIONS = BUNDLE["questions"]
    TEMPLATE = BUNDLE["template"]
    REASON_CODES = BUNDLE["reasons"]
pd.DataFrame(QUESTIONS)[["id", "question"]]

,id,question
0,V2Q001,How many activities are recorded in this snaps...
1,V2Q002,How many activities are marked humanitarian?
2,V2Q003,How many activities are recorded as completed?
3,V2Q004,How many activities have no actual start date?
4,V2Q005,How many activities have no actual end date?
5,V2Q006,How many activities have a planned start date?
6,V2Q007,How many activities have a planned end date?
7,V2Q008,How many activities use GBP as their default c...
8,V2Q009,How many distinct organisation references are ...
9,V2Q010,How many distinct named locations are recorded?


## 2. Graph schema as the agent sees it (raw codes, `source_id` on everything)

In [3]:
SCHEMA = '''
Graph loaded by fa26/retrieval/load_neo4j.py (schema_prototype.py). Every node has source_id = '<table>.csv:row:<n>';
every relationship has source_ids = a LIST of such ids (duplicate CSV rows collapse into one edge). Empty values are null, never ''.
  (:Activity {iati_identifier, publisher, title, description, status, status_name, match_basis, humanitarian,
              start_planned, start_actual, end_planned, end_actual, start_date, end_date, last_updated, default_currency, hotspot_countries, source_id})
      publisher slug: 'oxfamgb' | 'oxfamus' | 'wfp'.  status: '2' implementation, '3' finalisation (completed).  humanitarian: '1' | '0' | null.
      start_actual/start_planned/end_actual/end_planned are Date or null; start_date = start_actual else start_planned (Date, never null).
      last_updated is DateTime. Compare with date('2020-01-01'); a.start_actual.year gives the year.
  (:Organisation {ref, name, type, publisher, source_id})           // ref is the IATI identifier; refs without a real id start with 'name:'
  (:Country {code, name, hotspot_tier, source_id})                  // code ISO-2, e.g. 'SO'
  (:Sector {sector_key, code, vocabulary, name, is_hunger, source_id})
  (:Transaction {tx_id, type, type_name, date, value, currency, value_date, source_id})
      type is a code string: '1' incoming funds, '2' outgoing commitment, '3' disbursement, '4' expenditure.
      value is a signed float; date is Date or null. currency 'USD' | 'GBP' | ...
  (:Budget {budget_id, type, status, type_name, status_name, period_start, period_end, value, currency, source_id})   // value float, type/status code strings
  (:Location {location_id, name, lat, lon, source_id})
  (:SDG {sdg_key, code, name, source_id})

Relationships (each with source_ids list):
  (:Organisation)-[:REPORTS]->(:Activity)
  (:Organisation)-[:PARTICIPATES_IN {role, role_name}]->(:Activity)     // role code string '1' funding .. '4' implementing
  (:Activity)-[:TARGETS {percentage, is_hotspot}]->(:Country)           // is_hotspot is boolean
  (:Activity)-[:CLASSIFIED_AS {percentage}]->(:Sector)
  (:Activity)-[:LOCATED_AT]->(:Location)
  (:Activity)-[:CONTRIBUTES_TO]->(:SDG)
  (:Activity)-[:HAS_TRANSACTION]->(:Transaction)
  (:Activity)-[:HAS_BUDGET]->(:Budget)
  (:Organisation)-[:PROVIDES]->(:Transaction)          // the funder
  (:Transaction)-[:RECEIVED_BY]->(:Organisation)

Publisher slugs map to Organisation.ref: oxfamgb = 'GB-CHC-202918', oxfamus = 'US-EIN-23-7069110', wfp = 'XM-DAC-41140'.
Money: sum Transaction.value only within the type, currency and date range the question specifies; never convert currencies.
Negative transactions are their own category: a 'positive total' sums value > 0 only, a 'negative total' sums value < 0 only, 'net' sums both.
Evidence: collect(DISTINCT a.source_id) for nodes and reduce(acc = [], l IN collect(DISTINCT r.source_ids) | acc + coalesce(l, [])) for relationships.
Never put iati_identifier, ref or location_id in evidence.
'''
LABEL_TABLE = {"Activity": "nodes_activities.csv", "Organisation": "nodes_organisations.csv", "Country": "nodes_countries.csv",
               "Sector": "nodes_sectors.csv", "Transaction": "nodes_transactions.csv", "Budget": "nodes_budgets.csv",
               "Location": "nodes_locations.csv", "SDG": "nodes_sdgs.csv"}

## 3. Models (from `.env`; any `<PROVIDER>_MODEL` or `<PROVIDER>_MODEL_<SUFFIX>` registers one model)

In [4]:
# ---- model switch: edit this line, never .env. None = every model registered in .env.
# Command line override without editing anything: IATI_RUN_MODELS="anthropic/claude-sonnet-4-6,zai/glm-5.3"
RUN_MODELS = ["anthropic/claude-sonnet-4-6", "zai/glm-5.3"]

def env(k, default=None): return os.environ.get(k, default) or default

MODELS = []
if env("ANTHROPIC_API_KEY"):
    for m in (env("ANTHROPIC_EVAL_MODELS", "") or "").split(","):
        if m.strip(): MODELS.append({"name": f"anthropic/{m.strip()}", "provider": "anthropic", "model": m.strip()})
for prov in ("OPENAI", "GROQ", "NVIDIA", "ZAI", "DASHSCOPE", "DEEPSEEK", "MOONSHOT", "OPENROUTER"):
    if not env(f"{prov}_API_KEY"): continue
    url = env(f"{prov}_BASE_URL")
    if prov != "OPENAI" and not (url and url.startswith("https://")):
        print(f"skip {prov}: HTTPS {prov}_BASE_URL required"); continue
    for k, v in sorted(os.environ.items()):
        if (k == f"{prov}_MODEL" or k.startswith(f"{prov}_MODEL_")) and v.strip():
            MODELS.append({"name": f"{prov.lower()}/{v.strip()}", "provider": "openai_compat", "model": v.strip(),
                           "key_env": f"{prov}_API_KEY", "base_url": url})
if DRY_RUN:
    MODELS = [{"name": "fake/abstain-all", "provider": "fake"}]
assert MODELS, "No models registered: check ANTHROPIC_EVAL_MODELS / *_MODEL in fa26/.env"
_sel = [m.strip() for m in os.environ.get("IATI_RUN_MODELS", "").split(",") if m.strip()] or RUN_MODELS
if _sel and not DRY_RUN:
    _missing = sorted(set(_sel) - {m["name"] for m in MODELS}); assert not _missing, f"RUN_MODELS not registered in .env: {_missing}"
    MODELS = [m for m in MODELS if m["name"] in _sel]
pd.DataFrame(MODELS)[["name", "provider"]]

,name,provider
0,anthropic/claude-sonnet-4-6,anthropic
1,zai/glm-5.3,openai_compat


In [5]:
def make_llm(spec):
    if spec["provider"] == "anthropic":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=spec["model"], temperature=0, max_tokens=2000, timeout=120)
    if spec["provider"] == "openai_compat":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=spec["model"], api_key=os.environ[spec["key_env"]], base_url=spec["base_url"],
                          temperature=0, max_tokens=2000, timeout=180)
    from langchain_core.language_models.fake_chat_models import FakeListChatModel
    return FakeListChatModel(responses=[json.dumps({"status": "abstain", "reasons": ["NEEDS_DATA_MISSING"],
                                                    "explanation": "dry run", "cypher": ""})] * 40)

## 4. Neo4j executor (read-only, guarded)

In [6]:
if DRY_RUN:
    def run_cypher(q): return []
    print("dry run: no database")
else:
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))
    driver.verify_connectivity()
    with driver.session(database=env("NEO4J_DATABASE", "neo4j")) as s:
        nosid = s.run("MATCH ()-[r]->() WHERE r.source_ids IS NULL RETURN count(r) AS c").single()["c"]
    assert nosid == 0, "Graph has relationships without source_ids: reload with fa26/retrieval/load_neo4j.py (see notebooks/README.md)"
    run_cypher = read_executor(driver, env("NEO4J_DATABASE", "neo4j"), timeout=60, max_rows=20000)
    print("Neo4j connected, source_id verified")

Neo4j connected, source_id verified


## 5. The agent: plan → guard → execute → validate → format (LangGraph)

In [7]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage

class S(TypedDict, total=False):
    qid: str; question: str; contract: str; value_type: str
    raw: str; status: str; reasons: list; explanation: str; cypher: str
    error: Optional[str]; retries: int; rows: list; facts: dict; latency_ms: float; model_calls: int

SYSTEM = '''You are a data analyst answering questions about one fixed snapshot of IATI humanitarian aid data stored in Neo4j.
Reply with ONE JSON object only, no prose outside it:
{"status": "answer" | "abstain" | "clarify",
 "reasons": [],                      // for abstain/clarify: one or more of ''' + json.dumps(REASON_CODES) + '''
 "explanation": "...",               // one sentence
 "cypher": "..."}                    // for answer: one read-only Cypher query, else ""

Rules for status:
- "answer" when the question and its contract can be computed from the schema below.
- "abstain" when the snapshot cannot support the answer (e.g. needs-based rankings, historical versions, original XML fields). Use NEEDS_DATA_MISSING, HISTORY_UNAVAILABLE or RAW_XML_UNAVAILABLE.
- "clarify" when the question is underspecified (e.g. "total funding" without a financial measure, currency basis, or period). Use FUNDING_DEFINITION_REQUIRED, CURRENCY_BASIS_REQUIRED and/or PERIOD_REQUIRED.

Rules for the Cypher query:
- Return exactly three columns named key, value, evidence, one row per fact.
  key: the group label as a string (publisher slug, organisation ref, role code, match_basis, sdg_key), or exactly 'count' / 'total' when the contract asks for a single scalar.
  value: an integer from count(DISTINCT ...) or a number from sum(t.value); a date as toString(d) (first 10 characters).
  evidence: collect(DISTINCT x.source_id) for every contributing node plus the flattened source_ids of every contributing relationship
  (reduce(acc = [], l IN collect(DISTINCT r.source_ids) | acc + coalesce(l, []))). Never iati_identifier, ref or location_id.
- Use raw codes exactly as stored (strings): Transaction.type '1'..'4', PARTICIPATES_IN.role '1'..'4', Budget.type/status codes; is_hotspot is boolean true.
- Group transactions by their activity's publisher via (a:Activity)-[:HAS_TRANSACTION]->(t:Transaction); the funder is (o:Organisation)-[:PROVIDES]->(t).
- Compare dates as Date values: t.date >= date('2020-01-01') AND t.date < date('2021-01-01'); missing dates are null.
- For a total with no matching rows return one row with value 0 and an empty evidence list.
- Omit zero groups when the contract says so. No comments, no semicolons, no backticks, no CALL, no write clauses.
'''

if BUNDLE:
    SYSTEM += "\nPUBLIC BUNDLE INSTRUCTIONS:\n" + BUNDLE["instructions"] + "\n\n" + V2_REASON_GUIDE

def parse_plan(text):
    m = re.search(r"\{.*\}", text, re.S)
    obj = json.loads(m.group(0)) if m else {}
    status = obj.get("status", "answer")
    if status not in ("answer", "abstain", "clarify"): status = "answer"
    reasons = [r for r in obj.get("reasons", []) if r in REASON_CODES]
    cypher = (obj.get("cypher") or "").strip().strip("`")
    cypher = re.sub(r"^cypher\s*", "", cypher, flags=re.I).strip().rstrip(";").strip()
    return status, reasons, obj.get("explanation", ""), cypher

def fmt_value(v, kind):
    if kind == "integer": return str(int(round(float(v))))
    if kind == "decimal": return str(Decimal(str(v)).quantize(Decimal("0.01")))
    if kind == "date": return str(v)[:10]
    return str(v)

def build_agent(llm, run_cypher):
    def plan(st):
        human = f"Question {st['qid']}: {st['question']}\nContract: {st['contract']}\nExpected value type: {st['value_type']}"
        if st.get("error"):
            human += f"\n\nYour previous attempt failed: {st['error']}\nPrevious query:\n{st.get('cypher','')}\nReturn a corrected JSON object."
        out = llm.invoke([SystemMessage(content=SYSTEM + "\nSCHEMA:\n" + SCHEMA), HumanMessage(content=human)])
        text = out.content if isinstance(out.content, str) else "\n".join(b.get("text", "") for b in out.content if isinstance(b, dict))
        try:
            status, reasons, expl, cypher = parse_plan(text)
            if st["qid"].startswith("V2Q"): reasons = single_v2_reason(reasons, status)   # v2: exactly one reason code
            return {"raw": text, "status": status, "reasons": reasons, "explanation": expl, "cypher": cypher,
                    "error": None if (status != "answer" or cypher) else "empty cypher", "rows": [],
                    "model_calls": st.get("model_calls", 0) + 1}
        except Exception as e:
            return {"raw": text, "status": "answer", "cypher": "", "error": f"unparseable JSON: {type(e).__name__}", "rows": [],
                    "model_calls": st.get("model_calls", 0) + 1}

    def guard(st):
        try: guard_query(st["cypher"]); return {"error": None}
        except ValueError as e: return {"error": f"guard rejected query: {e}"}

    def execute(st):
        try: return {"rows": run_cypher(st["cypher"]), "error": None}
        except Exception as e: return {"rows": [], "error": f"execution error {type(e).__name__}: {str(e)[:300]}"}

    def validate(st):
        rows = st.get("rows") or []
        if not rows and st["value_type"] != "decimal":
            return {"error": "query returned no rows; check the filters and raw code values"}
        for r in rows:
            if set(r.keys()) != {"key", "value", "evidence"}:
                return {"error": f"columns must be exactly key, value, evidence; got {sorted(r.keys())}"}
            if r["value"] is None: return {"error": "value is null"}
            if not isinstance(r["evidence"], list): return {"error": "evidence must be a list from collect(DISTINCT x.source_id)"}
        ev_err = evidence_error(rows)          # output contract: evidence must be source ids, else retry with the reason
        if ev_err: return {"error": ev_err}
        return {"error": None}

    def format_(st):
        facts = {}
        if st["status"] == "answer" and not st.get("error"):
            rows = st.get("rows") or []
            if not rows:  # scalar total with no matches
                rows = [{"key": "total", "value": 0, "evidence": []}]
            for r in rows:
                ev = clean_evidence(r.get("evidence"))[0]            # never hand the scorer an invalid id
                if not ev:
                    ev = sorted(f"manifest:{t}" for l, t in LABEL_TABLE.items() if re.search(rf":{l}\b", st["cypher"]))
                facts[str(r["key"])] = {"value": fmt_value(r["value"], st["value_type"]), "evidence": ev}
            facts = normalise_facts(facts, st["question"], legacy=st["qid"].startswith("Q"))   # scalar key = count|total|date
        return {"facts": facts}

    def after_plan(st):     return "format" if st["status"] != "answer" else (("retry" if st.get("retries", 0) < 2 else "format") if st.get("error") else "guard")
    def after_step(st):     return ("retry" if st.get("retries", 0) < 2 else "format") if st.get("error") else "next"
    def bump(st):           return {"retries": st.get("retries", 0) + 1}

    g = StateGraph(S)
    for name, fn in [("plan", plan), ("guard", guard), ("execute", execute), ("validate", validate), ("format", format_), ("retry", bump)]:
        g.add_node(name, fn)
    g.set_entry_point("plan")
    g.add_conditional_edges("plan", after_plan, {"format": "format", "retry": "retry", "guard": "guard"})
    g.add_conditional_edges("guard", after_step, {"retry": "retry", "format": "format", "next": "execute"})
    g.add_conditional_edges("execute", after_step, {"retry": "retry", "format": "format", "next": "validate"})
    g.add_conditional_edges("validate", after_step, {"retry": "retry", "format": "format", "next": "format"})
    g.add_edge("retry", "plan"); g.add_edge("format", END)
    return g.compile()

_g = build_agent(make_llm({"provider": "fake"}), lambda q: []).get_graph()
print("nodes:", [n for n in _g.nodes if n not in ("__start__", "__end__")])
print("edges:", sorted({(e.source, e.target) for e in _g.edges}))

nodes: ['plan', 'guard', 'execute', 'validate', 'format', 'retry']
edges: [('__start__', 'plan'), ('execute', 'format'), ('execute', 'retry'), ('execute', 'validate'), ('format', '__end__'), ('guard', 'execute'), ('guard', 'format'), ('guard', 'retry'), ('plan', 'format'), ('plan', 'guard'), ('plan', 'retry'), ('retry', 'plan'), ('validate', 'format'), ('validate', 'retry')]


## 6. Run every model over the 20 questions and write benchmark submissions

In [8]:
records, submissions = [], []
for spec in MODELS:
    agent = build_agent(make_llm(spec), run_cypher)
    answers = []
    for q in QUESTIONS:
        t0 = time.time()
        try:
            st = agent.invoke({"qid": q["id"], "question": q["question"], "contract": q.get("contract", ""),
                               "value_type": q.get("value_type", "auto"), "retries": 0, "model_calls": 0})
        except Exception as e:
            st = {"status": "answer", "facts": {}, "reasons": [], "error": f"{type(e).__name__}: {str(e)[:200]}", "retries": 0}
        latency_ms = round((time.time() - t0) * 1000, 1)
        status = st.get("status", "answer")
        ans = {"id": q["id"], "status": status, "facts": st.get("facts", {}) if status == "answer" else {},
               "reasons": st.get("reasons", []) if status != "answer" else [], "latency_ms": latency_ms}
        if st.get("explanation"): ans["explanation"] = st["explanation"]
        answers.append(ans)
        records.append({"run": RUN_ID, "model": spec["name"], "qid": q["id"], "status": status, "n_facts": len(ans["facts"]),
                        "error": st.get("error"), "retries": st.get("retries", 0), "model_calls": st.get("model_calls", 0),
                        "latency_ms": latency_ms, "cypher": st.get("cypher")})
        print(f"{spec['name']:<28} {q['id']} {status:<8} facts={len(ans['facts']):<3} retries={st.get('retries',0)} {latency_ms:>8.0f} ms  {('ERR ' + st['error'][:70]) if st.get('error') else ''}")
    run_name = f"{spec['name'].replace('/', '_')}__langgraph_plan-guard-execute__{RUN_ID}"
    path = PROCESSED / f"05_submission_{spec['name'].replace('/', '_')}_{RUN_ID}.json"
    save_json(path, submission(BUNDLE, run_name, SNAPSHOT_ID, answers, DRY_RUN))
    submissions.append(path); print("wrote", path.name)
results = pd.DataFrame(records)
results.to_csv(PROCESSED / f"05_live_eval_{RUN_ID}.csv", index=False)
results.groupby("model")[["retries", "model_calls", "latency_ms"]].mean().round(1)

anthropic/claude-sonnet-4-6  V2Q001 answer   facts=1   retries=0     3436 ms  


anthropic/claude-sonnet-4-6  V2Q002 answer   facts=1   retries=0     3119 ms  


anthropic/claude-sonnet-4-6  V2Q003 answer   facts=1   retries=0     2795 ms  


anthropic/claude-sonnet-4-6  V2Q004 answer   facts=1   retries=0     3292 ms  


anthropic/claude-sonnet-4-6  V2Q005 answer   facts=1   retries=0     3502 ms  


anthropic/claude-sonnet-4-6  V2Q006 answer   facts=1   retries=0     2918 ms  


... (71 outputs left out)

zai/glm-5.3                  V2Q073 abstain  facts=0   retries=0     3063 ms  


zai/glm-5.3                  V2Q074 abstain  facts=0   retries=0     2924 ms  


zai/glm-5.3                  V2Q075 abstain  facts=0   retries=0    15467 ms  
wrote 05_submission_zai_glm-5.3_2026-11-05_193242_484749.json


,retries,model_calls,latency_ms
model,,,
anthropic/claude-sonnet-4-6,0.1,1.1,3873.3
zai/glm-5.3,0.0,1.0,27958.5


## 7. Score with the benchmark's own scorer (plus the SQL baseline as the ceiling)

In [9]:
baseline = FA26 / "benchmark" / "runs" / "sqlite_baseline.json"
cmd = score_command(sys.executable, FA26, submissions, RESULTS, BUNDLE)
proc = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
print(proc.stdout); print(proc.stderr[-2000:] if proc.returncode else "")
assert proc.returncode == 0, "scorer rejected a submission; see stderr above"

[
  {
    "run_name": "anthropic_claude-sonnet-4-6__langgraph_plan-guard-execute__2026-11-05_193242_484749",
    "synthetic": false,
    "summary": {
      "questions": 40,
      "correctness_pct": 92.5,
      "evidence_f1_pct": 84.87,
      "behavior_pct": 90.0,
      "overall_pct": 90.6,
      "latency_samples": 40,
      "mean_latency_ms": 3873.3375
    },
    "cost_usd": null,
    "by_category": {
      "direct": {
        "questions": 10,
        "correctness_pct": 90.0,
        "evidence_f1_pct": 90.0,
        "behavior_pct": null,
        "overall_pct": 90.0,
        "latency_samples": 10,
        "mean_latency_ms": 2933.29
      },
      "financial": {
        "questions": 7,
        "correctness_pct": 100.0,
        "evidence_f1_pct": 92.59,
        "behavior_pct": null,
        "overall_pct": 97.78,
        "latency_samples": 7,
        "mean_latency_ms": 3664.8714285714286
      },
      "limitations": {
        "questions": 5,
        "correctness_pct": 100.0,
        "evid

In [10]:
details = read_scores(RESULTS, BUNDLE)
rows = [{"model": r["summary"]["run_name"].split("__")[0], "qid": q["id"], "category": q["category"],
         "score": round(q["score"], 2), "correctness": round(q["correctness"], 2),
         "evidence_f1": None if q["evidence_f1"] is None else round(q["evidence_f1"], 2),
         "behavior": None if q["behavior"] is None else round(q["behavior"], 2)}
        for r in details["runs"] for q in r["questions"]]
per_q = pd.DataFrame(rows)
matrix = per_q.pivot(index="qid", columns="model", values="score")
display(matrix)
print("\nhardest questions (mean score across live models, excluding the SQL baseline):")
live = per_q[~per_q.model.str.startswith("sqlite")]
print(live.groupby(["qid", "category"]).score.mean().sort_values().head(8).to_string())

model,anthropic_claude-sonnet-4-6,zai_glm-5.3
qid,,
V2Q001,1.00,1.00
V2Q002,1.00,1.00
V2Q003,1.00,1.00
V2Q004,1.00,1.00
V2Q005,1.00,1.00
V2Q006,1.00,1.00
V2Q007,1.00,1.00
V2Q008,1.00,1.00
V2Q009,1.00,1.00



hardest questions (mean score across live models, excluding the SQL baseline):
qid     category    
V2Q026  two_step        0.40
V2Q042  organisation    0.50
V2Q010  direct          0.50
V2Q023  two_step        0.90
V2Q025  two_step        0.90
V2Q027  two_step        0.92
V2Q058  financial       0.93
V2Q049  organisation    0.95


## 8. How to read the result and what comes next

- **Overall %** in `comparison.md` is the number to report: 70% value correctness + 30% evidence lineage for Q01 to Q16, reason-code F1 for Q17 to Q20. The SQL baseline is 100 by construction and marks the ceiling of this snapshot.
- **Correctness without evidence** (a model that gets values right but cites nothing) caps at 76%, exactly the `correct_values_without_evidence_control` run in `fa26/benchmark/results/`. Evidence F1 is where a graph agent should beat a plain SQL-free LLM, because `collect(DISTINCT source_id)` is cheap in Cypher.
- Typical failure modes to look for in the CSV: `guard rejected` (model wrote comments or CALL), `columns must be exactly` (model ignored the key/value/evidence contract), `execution error` (raw codes used as integers, dates compared as dates), abstaining on an answerable question (scores 0), answering a limitation question (scores 0).
- Next: (a) feed each model's failures back as few-shot examples and re-run; (b) swap this deterministic graph for a ReAct agent with Zhelin's parameterised query tools and compare on the same 20 questions; (c) when the hunger filter changes the dataset, Deniz re-pins a snapshot v2 and all runs repeat.